In [ ]:
import pandas as pd
import os
import re

In [ ]:
def get_csv_files(path):
    pattern = re.compile(r'.*\.csv$')
    matching_files = []
    for root, _, files in os.walk(path):
        for filename in files:
            if pattern.match(filename):
                absolute_path = os.path.join(root, filename)
                matching_files.append(absolute_path)
    return matching_files

In [ ]:
def match_dfs_and_names(files):
    #get files in matched groups
    ring_dfs_files = [file for file in files if 'measurements_' in os.path.basename(file)]
    node_dfs_files = [file for file in files if 'Node' in os.path.basename(file)]
    skeleton_dfs_files = [file for file in files if 'skeleton' in os.path.basename(file)]
    #get names of files
    ring_names = list(map(os.path.basename,ring_dfs_files))
    node_names = list(map(os.path.basename,node_dfs_files))
    skeleton_names = list(map(os.path.basename,skeleton_dfs_files))
    #read in csv as pd
    ring_dfs = list(map(pd.read_csv,ring_dfs_files))
    node_dfs = list(map(pd.read_csv,node_dfs_files))
    skeleton_dfs = list(map(pd.read_csv,skeleton_dfs_files))
    ring_dfs_names = zip(ring_names,ring_dfs)
    node_dfs_names = zip(node_names,node_dfs)
    skeleton_dfs_names = zip(skeleton_names,skeleton_dfs)
    return ring_dfs_names, node_dfs_names, skeleton_dfs_names

def concat_and_save(zipped_names_dfs,path):
    dfs = []
    for name, df in zipped_names_dfs:
        if '50' in name:
            df['condition'] = '50mm'
        elif '90min' in name:
            df['condition'] = '300mm_90min'
        elif '6hr' in name:
            df['condition'] = '300mm_6hr'
        else:
            print(f'condition could not be matched for {name}')
        dfs.append(df)
    merged_df = pd.concat(dfs,ignore_index=True)
    if 'measurements_' in name:
        merged_df.to_csv(os.path.join(path,'All_Conditions_Ring_Measurements.csv'))
    elif 'Node' in name:
        merged_df.to_csv(os.path.join(path,'All_Conditions_Node_Counts.csv'))
    elif 'skeleton' in name:
        merged_df.to_csv(os.path.join(path,'All_Conditions_Skeleton_Analysis.csv'))
    else:
        print("Could not get information from name")

In [ ]:
Oct_Data = r'Image_Data\Oct_2025_300mm_exp'
save_path = r'Output\Oct_2025_300mM_exp'
files = get_csv_files(Oct_Data)
ring_dfs_names, node_dfs_names, skeleton_dfs_names = match_dfs_and_names(files)
concat_and_save(ring_dfs_names,save_path)
concat_and_save(node_dfs_names,save_path)
concat_and_save(skeleton_dfs_names,save_path)
